 1. Membaca data hasil cleaning

In [1]:
import pandas as pd

df = pd.read_csv("data/processed/isp_bersih_berlabel.csv")
print(df.shape)
print(df["kualitas"].value_counts())
df[["download_median", "upload_median", "ping_median", "skor", "kualitas"]].head()

(98, 18)
kualitas
Baik      65
Sedang    21
Buruk     12
Name: count, dtype: int64


,download_median,upload_median,ping_median,skor,kualitas
0,369.36,29.99,32.0,6,Baik
1,258.63,151.68,25.5,6,Baik
2,192.30,42.50,30.0,6,Baik
3,270.12,33.45,28.0,6,Baik
4,211.26,67.07,28.0,6,Baik



2. Deteksi outlier (IQR)

In [2]:
for m in ["download_median", "upload_median", "ping_median"]:
    q1, q3 = df[m].quantile([0.25, 0.75])
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[m] < bawah) | (df[m] > atas)).sum()
    print(f"{m}: {n} outlier (batas {bawah:.1f} sampai {atas:.1f})")

download_median: 2 outlier (batas -245.8 sampai 579.4)
upload_median: 6 outlier (batas -58.2 sampai 141.7)
ping_median: 11 outlier (batas -16.1 sampai 106.9)


3. Keputusan outlier



Outlier dipertahankan karena merupakan data asli (ISP fiber berkecepatan tinggi dan ISP Indonesia dengan ping tinggi), dan Decision Tree tidak sensitif terhadap outlier.

4. Seleksi fitur

In [3]:
fitur = ["sample_count",
         "download_average", "download_p25", "download_p75",
         "upload_average",   "upload_p25",   "upload_p75",
         "ping_average",     "ping_p25",     "ping_p75"]
X, y = df[fitur], df["kualitas"]
print("Jumlah fitur:", X.shape[1])

Jumlah fitur: 10


Median tidak dipakai sebagai fitur karena dipakai untuk membuat label (mencegah kebocoran label

5. Pembagian data latih dan uji

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nKelas data latih:\n", y_train.value_counts())
print("\nKelas data uji:\n", y_test.value_counts())

Train: (78, 10) | Test: (20, 10)

Kelas data latih:
 kualitas
Baik      52
Sedang    17
Buruk      9
Name: count, dtype: int64

Kelas data uji:
 kualitas
Baik      13
Sedang     4
Buruk      3
Name: count, dtype: int64


6. Menyimpan hasil

In [6]:
train = X_train.copy(); train["kualitas"] = y_train
test = X_test.copy();   test["kualitas"] = y_test
train.to_csv("data/processed/data_latih.csv", index=False)
test.to_csv("data/processed/data_uji.csv", index=False)
print("Tersimpan")

Tersimpan


7. Teknik yang tidak diperlukan

Imputasi tidak diperlukan (tidak ada missing value), encoding tidak diperlukan (semua fitur numerik), scaling tidak diperlukan (Decision Tree tidak sensitif terhadap skala fitur).